# Logistic Regression Task

Task 2: Run a Logistic regression in your pairs


Use the titanic sample dataset from seaborn
Predict: Survival rate based on available variables
1. Get & understand the data
 Understand what each passenger variable represents and establish what you're trying to predict.
2. Explore & clean the data
 Check missing values, duplicates, unusual values, categorical variables and class balance.
3. Choose X and y variables
4. Check initial logistic regression assumptions
5. Split into training & test data
6. Create & train the model
7. Make predictions
8. Evaluate model performance
 Confusion matrix
 Accuracy
 Precision
 Recall
 F1 score
 ROC/AUC where appropriate
9. Check model diagnostics & remaining assumptions
10. Interpret, improve & retest the model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay, classification_report)



In [ ]:
df = sns.load_dataset('titanic')

In [ ]:
df.head()

In [ ]:
print(df.shape)

In [ ]:
df.describe()

In [ ]:
print("duplicate rows: " , df.duplicated().sum())
print(df.isna().sum())

In [ ]:
df.groupby("sex")["survived"].mean()

In [ ]:
df.groupby("pclass")["survived"].mean()

In [ ]:
df = df.drop_duplicates()

df.duplicated().sum()

In [ ]:
df = df.drop(columns="deck")

In [ ]:
features = ["sex", "age", "pclass", "parch", "fare"]
X = df[features]
Y = df["survived"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)



In [ ]:
# fill missing ages using the TRAINING median only
age_median = X_train["age"].median()
X_train = X_train.assign(age=X_train["age"].fillna(age_median))
X_test = X_test.assign(age=X_test["age"].fillna(age_median))

# turn text categories into 0/1 columns
X_train = pd.get_dummies(X_train, columns=["sex", "pclass"], drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=["sex", "pclass"], drop_first=True, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.columns.tolist())

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)

prediction = model.predict(X_test)                 # 0 or 1
probability = model.predict_proba(X_test)[:, 1]    # chance of survival

pd.DataFrame({"actual": Y_test.values, "predicted": prediction,
              "p_survive": probability.round(3)}).head(10)

In [ ]:
print(confusion_matrix(Y_test, prediction))
print(classification_report(Y_test, prediction))
print("AUC:", roc_auc_score(Y_test, probability))

ConfusionMatrixDisplay.from_predictions(Y_test, prediction)
RocCurveDisplay.from_estimator(model, X_test, Y_test)
plt.show()